# Klassifikation - Mindestanforderungen 6
- Führen Sie mit dem Algorithmus Ihrer Wahl eine Klassifikationsaufgabe auf Ihren Daten durch.
    - Ziel war die Vorhersage der Zielvariable `Employment`
    - Klassen waren _employed_, _independent contractor, freelancer, or self-employed_, _student_ und _not employed_
    - unerwünschte Klassen wurden entfernt
    - Wir haben die **lineare Support Vector Machine** genutzt
    - Einsatz in der Pipeline mit Text- und numerischen Features
___
- Teilen Sie dazu zunächst die Daten auf, um Overfitting beim Trainieren des Algorithmus und bei der Parameterauswahl zu vermeiden. Erklären Sie die gewählte Strategie und die Größenverhältnisse.
    - Wir haben auf 80/20 gesplittet, also 80% Train und 20% Test (`train_test_split(test_size=0.2)`)
    - wir haben `stratify=y` angewandt um eine ähnliche Klassenverteilung in den Train- und Testdaten zu erhalten
    - um Overfitting bei Parametern zu verhindern haben wir Parameter-Tuning nur auf das Trainingsset mit 3-facher Cross-Validation angewandt
---
- Wählen Sie geeignete Features aus und setzen Sie die Parameter des Algorithmus. Beschreiben Sie das gewälhte Vorgehen für die Auswahl der Features und Parameter. Berichten Sie den Parameterraum und die final gewählten Parameter. Geben Sie die Performanz auf den Trainingsdaten (bzw. Entwicklungsdaten, falls verwendet) an.
    - **Features**:
        - verbleibenden `object`-Spalten wurden pro Zeile zu einem Textfeld `__text__` zusammengeführt und per TF-IDF Vectorizer in numerische Merkmale transformiert
        - bei numerischen Spalten wurden fehlende Werte mit dem Median aufgefüllt und anschließend Standardisiert
        - Entfernt wurden: `ResponseId`, `AgeNum`, `RemoteCategoryNum` und `ConvertedCompTotal` (Gehälter)
    - **Parameterauswahl**
        - `GridSearchCV` (cv=3) auf Trainingsdaten
        - Optimierungsmaß: macro-F1 (wegen Klassenunwucht)
    - **Parameterraum**:
        - `ngram_range`: (1,1), (1,2)
        - `min_df`: 5, 10
        - `max_df`: 0.9
        - `C`: 0.5, 1.0, 2.0
        - `class_weight`: None, balanced
    - **Beste Parameter**:
        - `ngram_range` = (1,2)
        - `min_df` = 5
        - `max_df` = 0.9
        - `C` = 0.5
        - `class_weight` = balanced
    - **Train-Leistung**
        - CV macro-F1 ~ 0,685
---
- Evaluieren Sie die Klassifikation auf den ungesehenen Testdaten. Betrachten Sie Precision und Recall sowie den F-Wert. Welches Maß ist für Ihre Anwendung wichtiger? Bewerten Sie Ihr Ergebnis. Ist es in der Praxis voraussichtlich zufriedenstellend?
    - Test-Ergebnisse
        - Accuracy: 0.94
        - macro-F1: 0.74
    - Precision/Recall
        - sehr hoch für _employed_ (F1: 0,97)
        - gut für _self-employed_ (F1: 0,80)
        - geringer für _student_ (F1: 0,61) und not _employed_ (F1: 0,59) da es kleine Klassen sind
    - wichtigstes Maß
        - macro-F1 ist wichtiger als die Accuracy, da die Klassen sehr stark unbalanciert sind
    - Bewertung
        - das Modell erkennt Mehrheitsklassen sehr zuverlässig
        - Minderheitsklassen werden schwieriger erkannt, hier wären in der Praxis voraussichtlich weitere Maßnahmen nötig

# Codeerklärungen
- imports laden
- csv einlesen
- Zielvariable `Employment` festlegen, die durch Klassifikation vorhergesagt werden soll

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

In [23]:
df = pd.read_csv("survey_results_cleaned_final.csv")
target_col = "Employment"

FileNotFoundError: [Errno 2] No such file or directory: '../survey_results_cleaned_final.csv'

- Zeilen ohne `Employment` werden entfernt
- sehr seltene Klassen werden entfernt, da sie irrelevant sind und zu wenige Beispiele haben (`prefer not to say` & `Other`
- zudem wird `ResponseId` entfernt, da es nur eine ID ist - kein inhaltlicher Mehrwert
- `AgeNum` ist redundant aufgrund von `Age` (Textspalte)
- `ConvertedCompTotal` zu viele leere Zellen/NaN-Werte

In [13]:
df = df.dropna(subset=[target_col]).copy()
df = df[df[target_col] != "i prefer not to say"].copy()
df = df[df[target_col] != "Other"].copy()

df = df.drop(columns=["ResponseId", "AgeNum", "ConvertedCompTotal" , "RemoteCategoryNum"], errors="ignore")

- Nach `object`-Spalten suchen, da sie meist Text/Kategorien sind
- `Employment` muss entfernt werden, da sonst geschummelt werden würde
- Aus allen Textspalten eine gemeinsamen Text pro Zeile (`__text__`)
#### Warum?
- kategoriale Spalten als Textklassifikation zu behandeln
- Danach kann TFIDF Vectorizer wie beim Clustering numerische Features daraus machen

In [14]:
text_cols = df.select_dtypes(include=["object"]).columns.tolist()
text_cols = [c for c in text_cols if c not in {target_col, "cluster"}]
df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

- numerische Spalten sammeln
- Auch hier wird Zielvariable `Employment` ausgeschlossen

In [24]:
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
num_cols = [c for c in num_cols if c not in {target_col, "cluster"}]

- `x` enthält Features (Text und Numerisch)
- `y` enthält die Labels, also die Employment-Klassen
- Ausgabe der Shapes und Klassenverteilung um Werte zu überprüfen

In [25]:
X = df[["__text__"] + num_cols].copy()
y = df[target_col].astype(str).copy()

print("X shape:", X.shape, "| y shape:", y.shape)
print("Target distribution:\n", y.value_counts())

X shape: (18594, 6) | y shape: (18594,)
Target distribution:
 Employment
employed                                                15779
independent contractor, freelancer, or self-employed     2118
student                                                   487
not employed                                              210
Name: count, dtype: int64


- wir splitten in 80% Training und 20% Test
- `stratify=y` -> Klassenverteilung bleibt bei Train und Test ungefähr gleich
- stratify wichtig, weil Zielvariable stark unbalanciert ist

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Textspalten
- `TfidfVectorizer` macht aus Wörtern Zahlen
- dabei werden Wörter so gewichtet, dass häufige "Standardwörter" weniger wichtig sind und informative Wörter stärker zählen
- `max_features=20000` begrenzt die Anzahl der Textfeatures, damit die Werte nicht explodieren
- `sublinear_tf=True` dämpft extrem häufige Wörter zusätzlich

### Numerische Spalten
- `SimpleImputer(median)` füllt fehlende numerische Werte, also `NaN-Werte`, mit dem Median der Spalte
- StandardScaler skaliert die Zahlen auf vergleichbare Größenordnungen, was Support Vector Machines hilft, da sie empfindlich auf unterschiedliche Skalen reagieren können

In [27]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(sublinear_tf=True, max_features=20000), "__text__"),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols)
    ],
    remainder="drop"
)

- in der Pipeline kommt nun die Vorverarbeitung und der Klassifikator (LinearSVC) zusammen
- Warum LinearSVC?
    - für Textdaten mit vielen Features funktioniert lineare SVC oft sehr gut

In [28]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LinearSVC(max_iter=20000))
])

- hier werden mehrere sinnvolle Einstellungen getestet:
- `ngram_range`:
    - (1,1) = nur einzelne Wörter
    - (1,2) = Wörter + Wortpaare -> Kontext besser zu erfassen
- `min_df`:
    - ignoriert sehr seltene Wörter (erzeugt oft nur rauschen)
- `C`:
    - Regularisierung der SVM: kleiner C entspricht stärkerer Regularisierung, ein größeres C bedeutet mehr Flexibilität
- `class_weight="balanced"`:
    - wichtig bei unbalancierten Klassen, da so kleine Klassen stärker gewichtet werden

In [29]:
param_grid = {
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__min_df": [5, 10],
    "preprocessing__text__max_df": [0.9],
    "classifier__C": [0.5, 1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

- GridSearchCV testet alle Parameterkombinationen
- Bewertung: `f1_macro`
    - sinnvoll, weil jede Klasse gleich gewichtet wird
- cv=3 bedeutet 3-fache Cross-Validation auf dem Trainingsset
- Testset bleibt unangetastet -> fairer Vergleich

In [30]:
grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("\nBest CV macro-F1:", grid.best_score_)
print("Best params:", grid.best_params_)

Fitting 3 folds for each of 24 candidates, totalling 72 fits

Best CV macro-F1: 0.685485059326809
Best params: {'classifier__C': 0.5, 'classifier__class_weight': 'balanced', 'preprocessing__text__max_df': 0.9, 'preprocessing__text__min_df': 5, 'preprocessing__text__ngram_range': (1, 2)}


- bestes Modell aus der GridSearch wird verwendet
- dann einmalige Evaluierung auf den Testdaten
- Confusion Matrix zeigt, welche Klassen miteinander verwechselt wurden
- Mehrheitsklasse (`employed`) wird sehr zuverlässig erkannt
- Minderheitsklassen werden häufiger fälschlich als `employed` vorhergesagt

In [22]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nTest report:\n", classification_report(y_test, y_pred, zero_division=0))

labels_sorted = best_model.named_steps["classifier"].classes_
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred, labels=labels_sorted))


Test report:
                                                       precision    recall  f1-score   support

                                            employed       0.96      0.98      0.97      3156
independent contractor, freelancer, or self-employed       0.85      0.76      0.80       424
                                        not employed       0.64      0.55      0.59        42
                                             student       0.62      0.61      0.61        97

                                            accuracy                           0.94      3719
                                           macro avg       0.77      0.72      0.74      3719
                                        weighted avg       0.94      0.94      0.94      3719

Confusion matrix:
 [[3082   41    6   27]
 [  96  321    3    4]
 [   2   12   23    5]
 [  31    3    4   59]]
